## DM-50837 Schema Updates

Eric Bellm

Determining which fields are currently populated.

In [1]:
import pandas as pd
import sqlite3
import numpy as np

In [2]:
from lsst.analysis.ap import ApdbPostgresQuery

In [4]:
# ap_verify
repo_base = '/sdf/group/rubin/u/ebellm/workspace/dc2/'
repo = f'{repo_base}/repo'
collection = 'ap_verify-output'
#butler = dafButler.Butler(repo, collection=collection)
connection = sqlite3.connect(f'{repo_base}/association.db')

# manual inspection of the association.db shows the following tables:

* DiaForcedSource            
* DiaSource
* DiaObject                  
* SSObject
* DiaObject_To_Object_Match  
* metadata

In [5]:
namespace = 'elhoward_dm49165_w_2025_19_comcam_appipe'
instrument = 'LSSTComCam'
apdb = ApdbPostgresQuery(namespace=namespace, instrument=instrument)

/opt/lsst/software/stack/conda/envs/lsst-scipipe-10.0.0/share/eups/Linux64/analysis_ap/g5e5941f2af+560a8a4b3b/python/lsst/analysis/ap/apdb.py:603: FutureWarning: The instrument name is now pulled from the APDB; this kwarg is ignored and will be removed after v29
  super().__init__(instrument=instrument, **kwargs)


In [6]:
apdb._tables.keys()

dict_keys(['DiaForcedSource', 'DiaObject', 'DiaSource', 'SSObject', 'DiaObject_To_Object_Match', 'metadata'])

Note that `ApdbPostgresQuery` has built-in methods for these queries that I'm sidestepping to ensure I'm getting what I want.  Unfortunately the collections and namespacing mean that apdb and sqlite apdbs don't get identical queries (which would be the reason to use to actual query interface).

In [7]:
with apdb.connection as connection:
    dia_forced_source = pd.read_sql_query(f'select * from {namespace}."DiaForcedSource" limit 10000;', connection)
dia_forced_source.head()

,diaForcedSourceId,diaObjectId,ra,dec,visit,detector,psfFlux,psfFluxErr,midpointMjdTai,scienceFlux,scienceFluxErr,band,time_processed,time_withdrawn
0,23859435340300289,23859435206083089,53.082600,-28.159186,2024110800246,0,986.2405,145.24733,60623.259329,1381.2434,144.16231,r,2025-05-18 02:25:01.627000+00:00,None
1,23859435341348865,23859435207131581,53.124770,-27.739816,2024110800246,2,5592.9175,226.03679,60623.259329,51281.4530,226.47531,r,2025-05-18 02:27:52.873000+00:00,None
2,23859435341873153,23859435207655444,53.175112,-28.339939,2024110800246,3,2307.8425,306.84210,60623.259329,84104.7500,305.61690,r,2025-05-18 02:29:19.473000+00:00,None
3,23859435341873154,23859435207655456,53.206157,-28.346382,2024110800246,3,2978.1626,463.46484,60623.259329,235853.0600,460.98523,r,2025-05-18 02:29:19.473000+00:00,None
4,23859435342397441,23859435208179742,53.266141,-28.018474,2024110800246,4,777.8215,143.45383,60623.259329,1292.2717,142.34080,r,2025-05-18 02:30:43.516000+00:00,None


In [8]:
with apdb.connection as connection:
    dia_source = pd.read_sql_query(f'select * from {namespace}."DiaSource" limit 10000;', connection)
dia_source.head()

,diaSourceId,visit,detector,diaObjectId,ssObjectId,parentDiaSourceId,ssObjectReassocTime,midpointMjdTai,ra,raErr,...,pixelFlags_saturatedCenter,pixelFlags_suspect,pixelFlags_suspectCenter,pixelFlags_streak,pixelFlags_streakCenter,pixelFlags_injected,pixelFlags_injectedCenter,pixelFlags_injected_template,pixelFlags_injected_templateCenter,pixelId
0,23859435206082580,2024110800245,0,23859435206082580,0,0,None,60623.258521,52.901262,0.000013,...,False,False,False,False,False,False,False,False,False,9876589979065
1,23859435206082584,2024110800245,0,23859435206082584,0,0,None,60623.258521,52.905644,0.001609,...,False,False,False,False,False,False,False,False,False,9876590655270
2,23859435206082628,2024110800245,0,23859435206082628,0,0,None,60623.258521,52.904345,NaN,...,False,False,False,False,False,False,False,False,False,9876589860879
3,23859435206082630,2024110800245,0,23859435206082630,0,0,None,60623.258521,52.934727,0.000022,...,False,False,False,False,False,False,False,False,False,9876589035185
4,23859435206082650,2024110800245,0,23859435206082650,0,0,None,60623.258521,52.922970,0.000017,...,False,False,False,False,False,False,False,False,False,9876590338967


In [9]:
with apdb.connection as connection:
    dia_object = pd.read_sql_query(f'select * from {namespace}."DiaObject" where "validityEnd" is NULL limit 10000;', connection)
dia_object.head()

,diaObjectId,validityStart,validityEnd,ra,raErr,dec,decErr,ra_dec_Cov,radecMjdTai,pmRa,...,y_psfFluxMin,y_psfFluxMax,y_psfFluxStetsonJ,y_psfFluxLinearSlope,y_psfFluxLinearIntercept,y_psfFluxMaxSlope,y_psfFluxErrMean,lastNonForcedSource,nDiaSources,pixelId
0,23859435206082580,2025-05-18 02:14:32.283336+00:00,None,52.901262,0.000013,-28.284959,0.000010,-2.635811e-11,60623.258521,None,...,NaN,NaN,None,None,None,None,NaN,1970-01-01 00:00:00+00:00,1,9876589979065
1,23859435206082584,2025-05-18 02:14:32.283336+00:00,None,52.905644,0.001609,-28.268650,0.001205,-4.033811e-07,60623.258521,None,...,NaN,NaN,None,None,None,None,NaN,1970-01-01 00:00:00+00:00,1,9876590655270
2,23859435206082628,2025-05-18 02:14:32.283336+00:00,None,52.904345,NaN,-28.284903,NaN,NaN,60623.258521,None,...,NaN,NaN,None,None,None,None,NaN,1970-01-01 00:00:00+00:00,1,9876589860879
3,23859435206082650,2025-05-18 02:14:32.283336+00:00,None,52.922970,0.000017,-28.216926,0.000030,1.090848e-10,60623.258521,None,...,NaN,NaN,None,None,None,None,NaN,1970-01-01 00:00:00+00:00,1,9876590338967
4,23859435206082687,2025-05-18 02:14:32.283336+00:00,None,52.928494,0.002697,-28.200332,0.012372,3.101669e-05,60623.258521,None,...,NaN,NaN,None,None,None,None,NaN,1970-01-01 00:00:00+00:00,1,9876590826461


In [10]:
with apdb.connection as connection:
    ss_object = pd.read_sql_query(f'select * from {namespace}."SSObject"  limit 10000;', connection)
ss_object.head()

,ssObjectId,discoverySubmissionDate,firstObservationDate,arc,numObs,lcPeriodic,MOID,MOIDTrueAnomaly,MOIDEclipticLongitude,MOIDDeltaV,...,y_H,y_G12,y_HErr,y_G12Err,y_H_y_G12_Cov,y_Chi2,y_Ndata,maxExtendedness,minExtendedness,medianExtendedness


In [11]:
with apdb.connection as connection:
    dia_object_to_object = pd.read_sql_query(f'select * from {namespace}."DiaObject_To_Object_Match"  limit 10000;', connection)
dia_object_to_object.head()

,diaObjectId,objectId,dist,lnP


## Determine which fields are always NaN

In [12]:
def print_nan_columns(df):
    w_all_nan = df.isna().all(axis=0)
    return np.sort(df.columns[w_all_nan].to_list()).tolist()

In [13]:
print_nan_columns(dia_forced_source)

['time_withdrawn']

In [14]:
print_nan_columns(dia_source)

['dipoleAngleErr',
 'dipoleDec',
 'dipoleDecErr',
 'dipoleDec_dipoleAngle_Cov',
 'dipoleDec_dipoleLength_Cov',
 'dipoleFluxDiff_dipoleAngle_Cov',
 'dipoleFluxDiff_dipoleDec_Cov',
 'dipoleFluxDiff_dipoleLength_Cov',
 'dipoleFluxDiff_dipoleRa_Cov',
 'dipoleLengthErr',
 'dipoleLength_dipoleAngle_Cov',
 'dipoleLnL',
 'dipoleMeanFlux_dipoleAngle_Cov',
 'dipoleMeanFlux_dipoleDec_Cov',
 'dipoleMeanFlux_dipoleFluxDiff_Cov',
 'dipoleMeanFlux_dipoleLength_Cov',
 'dipoleMeanFlux_dipoleRa_Cov',
 'dipoleRa',
 'dipoleRaErr',
 'dipoleRa_dipoleAngle_Cov',
 'dipoleRa_dipoleDec_Cov',
 'dipoleRa_dipoleLength_Cov',
 'fpBkgd',
 'fpBkgdErr',
 'ixxErr',
 'ixx_ixy_Cov',
 'ixx_iyy_Cov',
 'ixyErr',
 'iyyErr',
 'iyy_ixy_Cov',
 'psfDec',
 'psfDecErr',
 'psfFlux_psfDec_Cov',
 'psfFlux_psfRa_Cov',
 'psfLnL',
 'psfRa',
 'psfRaErr',
 'psfRa_psfDec_Cov',
 'snapDiffFlux',
 'snapDiffFluxErr',
 'ssObjectReassocTime',
 'time_withdrawn',
 'trailAngleErr',
 'trailChi2',
 'trailDecErr',
 'trailDec_trailAngle_Cov',
 'trailDec

In [15]:
print_nan_columns(dia_object)

['g_fpFluxMean',
 'g_fpFluxMeanErr',
 'g_fpFluxSigma',
 'i_fpFluxMean',
 'i_fpFluxMeanErr',
 'i_fpFluxSigma',
 'nearbyExtObj1Sep',
 'nearbyExtObj2Sep',
 'nearbyExtObj3Sep',
 'nearbyLowzGal',
 'nearbyLowzGalSep',
 'nearbyObj1Dist',
 'nearbyObj1LnP',
 'nearbyObj2Dist',
 'nearbyObj2LnP',
 'nearbyObj3Dist',
 'nearbyObj3LnP',
 'parallax',
 'parallaxErr',
 'pmDec',
 'pmDecErr',
 'pmDec_parallax_Cov',
 'pmParallaxChi2',
 'pmParallaxLnL',
 'pmRa',
 'pmRaErr',
 'pmRa_parallax_Cov',
 'pmRa_pmDec_Cov',
 'r_fpFluxMean',
 'r_fpFluxMeanErr',
 'r_fpFluxSigma',
 'u_fpFluxMean',
 'u_fpFluxMeanErr',
 'u_fpFluxSigma',
 'validityEnd',
 'y_fpFluxMean',
 'y_fpFluxMeanErr',
 'y_fpFluxSigma',
 'y_psfFluxLinearIntercept',
 'y_psfFluxLinearSlope',
 'y_psfFluxMaxSlope',
 'y_psfFluxSigma',
 'y_psfFluxSkew',
 'y_psfFluxStetsonJ',
 'y_scienceFluxSigma',
 'z_fpFluxMean',
 'z_fpFluxMeanErr',
 'z_fpFluxSigma']

In [ ]:
# check on trail centers

In [17]:
dia_source['ra']

0       52.901262
1       52.905644
2       52.904345
3       52.934727
4       52.922970
          ...    
9995    53.165891
9996    53.249422
9997    53.249704
9998    53.298702
9999    53.303563
Name: ra, Length: 10000, dtype: float64

In [18]:
dia_source['trailRa']

0             NaN
1             NaN
2             NaN
3             NaN
4       52.923160
          ...    
9995    53.165734
9996    53.249408
9997    53.249771
9998    53.298598
9999          NaN
Name: trailRa, Length: 10000, dtype: float64

In [21]:
w = ~dia_source['trailRa'].isna()

In [23]:
np.sum(~np.isclose(dia_source.loc[w,'ra'], dia_source.loc[w,'trailRa']))

np.int64(26)

In [24]:
(dia_source.loc[w,'ra'] - dia_source.loc[w,'trailRa'])*3600

4      -0.683968
6       0.556638
7       0.748100
10     -0.026034
11      0.086047
          ...   
9994   -0.041660
9995    0.562458
9996    0.050350
9997   -0.241808
9998    0.373443
Length: 5875, dtype: float64